In [ ]:
import json
import os
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import traceback
load_dotenv()

%load_ext autoreload
%autoreload 2

root_path = Path().absolute().parent.parent
sys.path.append(str(root_path))
# print(f"Root path set to: {root_path}")
from eval_agent.user_agent.graph.user_agent_graph import build_graph 

In [ ]:

llm = os.getenv("LLM_MODEL")  # e.g., "gpt-4o"
provider = os.getenv("LLM_PROVIDER")  # e.g., "openai"
memory = True # True if it is the experiment of the agent with memory, False if it is the experiment of the agent without memory
database = os.getenv("EXPERIMENT_NAME")
print(f"Running evaluation for LLM: {llm}, Provider: {provider}, Memory: {memory}, Database: {database}")

# Load the experiment dataset
json_file = f'../dataset_generation/dialogue_dataset/{database}_dialogue_dataset.json'
with open(json_file, 'r', encoding='utf-8') as f:
    experiments = json.load(f)


# Build the evaluation graph with the specified memory and environment
eval_graph = build_graph(conversational_agent_memory=memory, env="tec")

experiment_memory = "memory" if memory else "agent_without_memory"
# Defines the name of the file where the results will be saved
evals_filename = f'{llm}-{experiment_memory}-{database}-eval.json'

# If the file already exists, load the experiments already evaluated
if os.path.exists(evals_filename):
    with open(evals_filename, 'r', encoding='utf-8') as f:
        evals = json.load(f)
else:
    evals = []

# Current template configuration (add details of the template used here)
processed_ids = {exp["experiment_id"] for exp in evals}

# Informations to Graph State
model_version = f"{provider}-{llm}"

experiment_type = f"text2sql_evaluation_{experiment_memory}"


Running evaluation for LLM: gpt-4o, Provider: azure, Memory: True, Database: mondial


TypeError: build_graph() got an unexpected keyword argument 'text_to_sql_agent_memory'

In [ ]:
from IPython.display import Image, display


try:
    display(Image(eval_graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
# bird_error_exps = ["6", "14", "24", "33", "34", "43"]
# Iterates over all experiments in the dataset
for experiment in experiments['dataset']:
    exp_id = experiment["experiment_id"]

    # if database == "bird":
        
    #     if exp_id in bird_error_exps:
    #         continue
    # Se o experimento já foi avaliado, pula para o próximo
    if exp_id in processed_ids:
        print(f"Pulando experimento {exp_id} já processado.")
        continue

    print(f"Processando experimento {exp_id}...")

    try:
        # Invoca a avaliação
        eval_result = eval_graph.invoke(
            {
                "experiment": experiment,
                "max_retries": 2,
                "debug_mode": True,
                "model_version": model_version,
                "experiment_type": experiment_type,
            },
            {"recursion_limit": 200}
        )

        print("Experimento avaliado:\n", eval_result["experiment_eval"])

        # Salva resultado
        evals.append({
            "experiment_id": exp_id,
            "experiment_config": eval_result.get(
                "experiment_config",
                {
                    "max_retries": 2,
                    "model_version": model_version,
                    "timestamp": datetime.now().isoformat(),
                    "experiment_type": experiment_type,
                }
            ),
            "experiment_eval": eval_result["experiment_eval"]
        })

    except Exception as e:
        print(f"\n Erro ao processar experimento {exp_id}: {e}")
        traceback.print_exc()

        # (Opcional) registra falha no arquivo de saída
        # evals.append({
        #     "experiment_id": exp_id,
        #     "experiment_config": {
        #         "model_version": model_version,
        #         "timestamp": datetime.now().isoformat(),
        #         "experiment_type": experiment_type,
        #     },
        #     "experiment_eval": None,
        #     "error": str(e)
        # })

    finally:
        # Atualiza o arquivo SEMPRE (sucesso ou erro)
        with open(evals_filename, 'w', encoding='utf-8') as f:
            json.dump(evals, f, indent=4)